# Deduplication Tasks
> Find and merge duplicate code across notebooks

In [ ]:
#| default_exp tasks.dedup

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
from __future__ import annotations
from typing import Any, Dict, List, Optional, Set, Tuple
from dataclasses import dataclass, field, asdict
from pathlib import Path
from collections import defaultdict
import difflib
import re
import hashlib
import time

from mcp.server.fastmcp import FastMCP

from nbdev_mcp.utils.paths import resolve_selector, settings_dict
from nbdev_mcp.utils.depgraph import (
    build_graph, DependencyGraph, SymbolNode, Edge, detect_duplicates
)
from nbdev_mcp.tasks.cache import ModidxCacheManager

## Data Structures

In [ ]:
#| export
@dataclass
class DuplicateGroup:
    """A group of duplicate/similar symbols.
    
    Attributes
    ----------
    canonical : str
        Recommended symbol to keep (the one with most imports).
    duplicates : List[str]
        Symbols to merge/remove.
    similarity_type : str
        One of: 'exact', 'functional', 'named', 'semantic'.
    similarity_score : float
        Similarity score (0.0-1.0).
    merge_strategy : str
        One of: 'keep_canonical', 'merge_superset', 'manual'.
    """
    canonical: str
    duplicates: List[str]
    similarity_type: str
    similarity_score: float
    merge_strategy: str = 'keep_canonical'
    
    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

In [ ]:
#| export
@dataclass
class DependentInfo:
    """Information about a symbol that depends on another.
    
    Attributes
    ----------
    symbol_id : str
        FQN of the dependent symbol.
    notebook : str
        Notebook containing the dependent.
    usage_type : str
        One of: 'import', 'call', 'inherit'.
    line_numbers : List[int]
        Line numbers where usage occurs.
    """
    symbol_id: str
    notebook: str
    usage_type: str
    line_numbers: List[int] = field(default_factory=list)
    
    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

In [ ]:
#| export
@dataclass
class MergeRoadmap:
    """Step-by-step plan to merge duplicates.
    
    Attributes
    ----------
    id : str
        Unique roadmap identifier.
    source_symbols : List[str]
        Symbols being merged.
    target_symbol : str
        Final merged symbol name.
    target_location : str
        Target module path.
    signature_changes : Dict[str, str]
        Old signature -> new signature mappings.
    dependents : List[DependentInfo]
        All affected code locations.
    steps : List[str]
        Human-readable migration steps.
    risk_level : str
        One of: 'low', 'medium', 'high'.
    breaking_changes : List[str]
        List of API incompatibilities.
    """
    id: str
    source_symbols: List[str]
    target_symbol: str
    target_location: str
    signature_changes: Dict[str, str] = field(default_factory=dict)
    dependents: List[DependentInfo] = field(default_factory=list)
    steps: List[str] = field(default_factory=list)
    risk_level: str = 'medium'
    breaking_changes: List[str] = field(default_factory=list)
    
    def to_dict(self) -> Dict[str, Any]:
        return {
            'id': self.id,
            'source_symbols': self.source_symbols,
            'target_symbol': self.target_symbol,
            'target_location': self.target_location,
            'signature_changes': self.signature_changes,
            'dependents': [d.to_dict() for d in self.dependents],
            'steps': self.steps,
            'risk_level': self.risk_level,
            'breaking_changes': self.breaking_changes
        }

## Duplicate Detection

In [ ]:
#| export
def find_functional_duplicates(
    graph: DependencyGraph,
    min_similarity: float = 0.8,
    deadline: Optional[float] = None
) -> List[DuplicateGroup]:
    """Find symbols with identical/similar behavior but different names.
    
    Uses AST hash comparison and structural similarity.
    
    Parameters
    ----------
    graph : DependencyGraph
        The dependency graph to analyze.
    min_similarity : float
        Minimum similarity threshold.
    deadline : float, optional
        Absolute time limit (time.monotonic), if provided.
    
    Returns
    -------
    List[DuplicateGroup]
        Groups of functionally duplicate symbols.
    """
    groups = []
    symbols = list(graph.symbols.values())
    
    # Group by hash for exact duplicates
    hash_groups: Dict[str, List[SymbolNode]] = defaultdict(list)
    for sym in symbols:
        if sym.source_hash and sym.line_count >= 3:
            hash_groups[sym.source_hash].append(sym)
    
    # Create groups for exact duplicates
    for hash_val, syms in hash_groups.items():
        if len(syms) > 1:
            # Choose canonical: most imports, then shortest name
            sorted_syms = sorted(
                syms,
                key=lambda s: (-len(graph.get_node_edges(s.id, 'out')), len(s.name))
            )
            groups.append(DuplicateGroup(
                canonical=sorted_syms[0].id,
                duplicates=[s.id for s in sorted_syms[1:]],
                similarity_type='exact',
                similarity_score=1.0,
                merge_strategy='keep_canonical'
            ))
    
    # Find similar (not exact) duplicates
    checked = set()
    for i, sym1 in enumerate(symbols):
        if deadline is not None and time.monotonic() > deadline:
            return groups
        if sym1.line_count < 5 or sym1.id in checked:
            continue
        
        similar = []
        for sym2 in symbols[i+1:]:
            if deadline is not None and time.monotonic() > deadline:
                return groups
            if sym2.line_count < 5 or sym2.id in checked:
                continue
            if sym1.kind != sym2.kind:
                continue
            if sym1.source_hash == sym2.source_hash:
                continue  # Already handled as exact
            
            # Compare normalized AST
            ratio = difflib.SequenceMatcher(
                None,
                SymbolNode._normalize_source(sym1.source),
                SymbolNode._normalize_source(sym2.source)
            ).ratio()
            
            if ratio >= min_similarity:
                similar.append((sym2, ratio))
        
        if similar:
            all_syms = [sym1] + [s[0] for s in similar]
            avg_score = sum(s[1] for s in similar) / len(similar)
            
            # Choose canonical
            sorted_syms = sorted(
                all_syms,
                key=lambda s: (-len(graph.get_node_edges(s.id, 'out')), len(s.name))
            )
            
            groups.append(DuplicateGroup(
                canonical=sorted_syms[0].id,
                duplicates=[s.id for s in sorted_syms[1:]],
                similarity_type='functional',
                similarity_score=avg_score,
                merge_strategy='merge_superset'
            ))
            
            for s in all_syms:
                checked.add(s.id)
    
    return groups

In [ ]:
#| export
def find_named_duplicates(
    graph: DependencyGraph,
    deadline: Optional[float] = None
) -> List[DuplicateGroup]:
    """Find symbols with same/similar names across modules.
    
    Detects:
    - Exact name matches in different modules
    - Private/public pairs (_name, name)
    - Near-matches (edit distance <= 2)
    - Common prefix/suffix variants (get_foo, fetch_foo)
    
    Parameters
    ----------
    graph : DependencyGraph
        The dependency graph to analyze.
    deadline : float, optional
        Absolute time limit (time.monotonic), if provided.
    
    Returns
    -------
    List[DuplicateGroup]
        Groups of similarly named symbols.
    """
    groups = []
    symbols = list(graph.symbols.values())
    
    # Group by base name (without leading underscores)
    name_groups: Dict[str, List[SymbolNode]] = defaultdict(list)
    for sym in symbols:
        if deadline is not None and time.monotonic() > deadline:
            return groups
        base_name = sym.name.lstrip('_')
        name_groups[base_name].append(sym)
    
    # Find exact name matches (different modules)
    for base_name, syms in name_groups.items():
        if deadline is not None and time.monotonic() > deadline:
            return groups
        if len(syms) > 1:
            # Check if they're in different modules
            modules = set(s.module for s in syms)
            if len(modules) > 1:
                sorted_syms = sorted(
                    syms,
                    key=lambda s: (-len(graph.get_node_edges(s.id, 'out')), s.name.count('_'))
                )
                groups.append(DuplicateGroup(
                    canonical=sorted_syms[0].id,
                    duplicates=[s.id for s in sorted_syms[1:]],
                    similarity_type='named',
                    similarity_score=1.0,
                    merge_strategy='manual'  # Name matches need review
                ))
    
    # Find private/public pairs in same module
    for sym in symbols:
        if deadline is not None and time.monotonic() > deadline:
            return groups
        if sym.name.startswith('_') and not sym.name.startswith('__'):
            public_name = sym.name[1:]
            public_id = f"{sym.module}.{public_name}"
            if public_id in graph.symbols:
                groups.append(DuplicateGroup(
                    canonical=public_id,
                    duplicates=[sym.id],
                    similarity_type='named',
                    similarity_score=0.95,
                    merge_strategy='keep_canonical'
                ))
    
    # Find prefix/suffix variants
    prefixes = ['get_', 'fetch_', 'load_', 'read_', 'compute_', 'calc_', 'find_', 'search_']
    checked = set()
    
    for sym1 in symbols:
        if sym1.id in checked:
            continue
        
        # Extract base without common prefixes
        base1 = sym1.name
        for prefix in prefixes:
            if base1.startswith(prefix):
                base1 = base1[len(prefix):]
                break
        
        if len(base1) < 3:
            continue
        
        variants = []
        for sym2 in symbols:
            if sym2.id == sym1.id or sym2.id in checked:
                continue
            
            base2 = sym2.name
            for prefix in prefixes:
                if base2.startswith(prefix):
                    base2 = base2[len(prefix):]
                    break
            
            if base1 == base2 and sym1.kind == sym2.kind:
                variants.append(sym2)
        
        if variants:
            all_syms = [sym1] + variants
            sorted_syms = sorted(
                all_syms,
                key=lambda s: (-len(graph.get_node_edges(s.id, 'out')), len(s.name))
            )
            groups.append(DuplicateGroup(
                canonical=sorted_syms[0].id,
                duplicates=[s.id for s in sorted_syms[1:]],
                similarity_type='named',
                similarity_score=0.85,
                merge_strategy='manual'
            ))
            for s in all_syms:
                checked.add(s.id)
    
    return groups

In [ ]:
#| export
def _get_embedding_device() -> str:
    """Detect best available device for embeddings."""
    try:
        import torch
        if torch.backends.mps.is_available():
            return 'mps'
        elif torch.cuda.is_available():
            return 'cuda'
    except ImportError:
        pass
    return 'cpu'


def find_semantic_duplicates(
    graph: DependencyGraph,
    min_similarity: float = 0.85,
    model_name: str = 'all-MiniLM-L6-v2',
    deadline: Optional[float] = None
) -> List[DuplicateGroup]:
    """Find symbols with similar purpose using embeddings.
    
    WARNING: This is expensive and should only be run with user confirmation.
    Uses local embeddings with MPS/CUDA acceleration when available.
    
    Parameters
    ----------
    graph : DependencyGraph
        The dependency graph to analyze.
    min_similarity : float
        Minimum cosine similarity threshold.
    model_name : str
        Sentence-transformers model name.
    deadline : float, optional
        Absolute time limit (time.monotonic), if provided.
    
    Returns
    -------
    List[DuplicateGroup]
        Groups of semantically similar symbols.
    """
    try:
        from sentence_transformers import SentenceTransformer
        import numpy as np
    except ImportError:
        return []  # sentence-transformers not installed
    
    if deadline is not None and time.monotonic() > deadline:
        return []

    device = _get_embedding_device()
    model = SentenceTransformer(model_name, device=device)
    
    # Prepare texts for embedding (docstring + function signature)
    symbols = [s for s in graph.symbols.values() if s.line_count >= 3]
    texts = []
    for sym in symbols:
        # Extract first docstring line if present
        doc = ''
        if '"""' in sym.source:
            match = re.search(r'"""([^"]+)', sym.source)
            if match:
                doc = match.group(1).strip().split('\n')[0]
        
        # Get signature (first line)
        sig = sym.source.split('\n')[0] if sym.source else ''
        
        texts.append(f"{sym.name}: {doc} {sig}")
    
    if not texts:
        return []
    
    # Generate embeddings
    embeddings = model.encode(texts, convert_to_numpy=True)
    
    # Compute pairwise similarities
    groups = []
    checked = set()
    
    for i, sym1 in enumerate(symbols):
        if deadline is not None and time.monotonic() > deadline:
            return groups
        if sym1.id in checked:
            continue
        
        similar = []
        for j, sym2 in enumerate(symbols[i+1:], i+1):
            if deadline is not None and time.monotonic() > deadline:
                return groups
            if sym2.id in checked:
                continue
            if sym1.kind != sym2.kind:
                continue
            
            # Cosine similarity
            sim = np.dot(embeddings[i], embeddings[j]) / (
                np.linalg.norm(embeddings[i]) * np.linalg.norm(embeddings[j])
            )
            
            if sim >= min_similarity:
                similar.append((sym2, float(sim)))
        
        if similar:
            all_syms = [sym1] + [s[0] for s in similar]
            avg_score = sum(s[1] for s in similar) / len(similar)
            
            sorted_syms = sorted(
                all_syms,
                key=lambda s: (-len(graph.get_node_edges(s.id, 'out')), len(s.name))
            )
            
            groups.append(DuplicateGroup(
                canonical=sorted_syms[0].id,
                duplicates=[s.id for s in sorted_syms[1:]],
                similarity_type='semantic',
                similarity_score=avg_score,
                merge_strategy='manual'  # Semantic matches need review
            ))
            
            for s in all_syms:
                checked.add(s.id)
    
    return groups


## Merge Planning

In [ ]:
#| export
def find_optimal_location(
    symbol_ids: List[str],
    graph: DependencyGraph
) -> str:
    """Find the lowest module that all dependents can access.
    
    Parameters
    ----------
    symbol_ids : List[str]
        Symbols to consider.
    graph : DependencyGraph
        The dependency graph.
    
    Returns
    -------
    str
        Optimal module path.
    """
    # Collect all modules that use these symbols
    dependent_modules: Set[str] = set()
    
    for sym_id in symbol_ids:
        for edge in graph.get_node_edges(sym_id, 'out'):
            if edge.target in graph.symbols:
                dependent_modules.add(graph.symbols[edge.target].module)
            elif edge.target in graph.notebooks:
                dependent_modules.add(graph.notebooks[edge.target].module)
    
    if not dependent_modules:
        # No dependents - keep in current location
        if symbol_ids and symbol_ids[0] in graph.symbols:
            return graph.symbols[symbol_ids[0]].module
        return ''
    
    # Find common prefix of all dependent modules
    module_parts = [m.split('.') for m in dependent_modules]
    
    # Find longest common prefix
    common = []
    for parts in zip(*module_parts):
        if len(set(parts)) == 1:
            common.append(parts[0])
        else:
            break
    
    # Use utils if no common prefix, otherwise the common ancestor
    if not common:
        return 'utils'
    
    return '.'.join(common)


def gather_dependents(
    symbol_ids: List[str],
    graph: DependencyGraph
) -> List[DependentInfo]:
    """Gather all dependents for a set of symbols.
    
    Parameters
    ----------
    symbol_ids : List[str]
        Symbols to check.
    graph : DependencyGraph
        The dependency graph.
    
    Returns
    -------
    List[DependentInfo]
        All dependents.
    """
    dependents = []
    seen = set()
    
    for sym_id in symbol_ids:
        for edge in graph.get_node_edges(sym_id, 'out'):
            if edge.target in seen:
                continue
            seen.add(edge.target)
            
            notebook = ''
            if edge.target in graph.symbols:
                notebook = graph.symbols[edge.target].notebook
            elif edge.target in graph.notebooks:
                notebook = edge.target
            
            dependents.append(DependentInfo(
                symbol_id=edge.target,
                notebook=notebook,
                usage_type=edge.kind
            ))
    
    return dependents

In [ ]:
#| export
def generate_merge_roadmap(
    group: DuplicateGroup,
    graph: DependencyGraph,
    target_module: Optional[str] = None
) -> MergeRoadmap:
    """Generate a merge roadmap for a duplicate group.
    
    Parameters
    ----------
    group : DuplicateGroup
        The duplicate group to merge.
    graph : DependencyGraph
        The dependency graph.
    target_module : str, optional
        Override target module location.
    
    Returns
    -------
    MergeRoadmap
        The merge plan.
    """
    all_symbols = [group.canonical] + group.duplicates
    
    # Determine target location
    if target_module:
        location = target_module
    else:
        location = find_optimal_location(all_symbols, graph)
    
    # Gather all dependents
    dependents = gather_dependents(all_symbols, graph)
    
    # Determine target symbol name
    canonical_sym = graph.symbols.get(group.canonical)
    target_name = canonical_sym.name if canonical_sym else group.canonical.split('.')[-1]
    
    # Build signature changes
    sig_changes = {}
    for dup_id in group.duplicates:
        dup_sym = graph.symbols.get(dup_id)
        if dup_sym:
            old_import = f"from {graph.lib_name}.{dup_sym.module} import {dup_sym.name}"
            new_import = f"from {graph.lib_name}.{location} import {target_name}"
            sig_changes[old_import] = new_import
    
    # Check for breaking changes
    breaking = []
    if group.similarity_score < 1.0:
        breaking.append(f"Similarity score {group.similarity_score:.2f} < 1.0 - review behavior differences")
    if group.merge_strategy == 'merge_superset':
        breaking.append("Merge requires combining functionality - test thoroughly")
    
    # Determine risk level
    risk = 'low'
    if len(dependents) > 10:
        risk = 'medium'
    if len(dependents) > 25 or breaking:
        risk = 'high'
    
    # Generate steps
    steps = [
        f"1. Create merge branch: git checkout -b merge/{target_name}",
        f"2. Verify canonical implementation at {group.canonical}",
    ]
    
    if group.similarity_score < 1.0:
        steps.append(f"3. Compare implementations - ensure {target_name} has superset of functionality")
    
    if location != (canonical_sym.module if canonical_sym else ''):
        steps.append(f"4. Move {target_name} to {location}")
    
    steps.append(f"{len(steps)+1}. Update {len(dependents)} import sites")
    
    for dup_id in group.duplicates:
        dup_sym = graph.symbols.get(dup_id)
        if dup_sym:
            steps.append(f"{len(steps)+1}. Remove duplicate: {dup_sym.module}.{dup_sym.name}")
    
    steps.append(f"{len(steps)+1}. Run nbdev_export && nbdev_test")
    steps.append(f"{len(steps)+1}. WAIT FOR HUMAN CONFIRMATION before merging")
    
    # Generate unique ID
    roadmap_id = hashlib.md5(
        f"{group.canonical}:{','.join(group.duplicates)}".encode()
    ).hexdigest()[:8]
    
    return MergeRoadmap(
        id=roadmap_id,
        source_symbols=all_symbols,
        target_symbol=target_name,
        target_location=location,
        signature_changes=sig_changes,
        dependents=dependents,
        steps=steps,
        risk_level=risk,
        breaking_changes=breaking
    )

## Main API

In [ ]:
#| export
def find_all_duplicates(
    project: Optional[str] = None,
    include_functional: bool = True,
    include_named: bool = True,
    include_semantic: bool = False,
    min_similarity: float = 0.8,
    timeout: Optional[int] = None
) -> Dict[str, Any]:
    """Find all duplicate symbols in a project.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias.
    include_functional : bool
        Include functionally duplicate code.
    include_named : bool
        Include similarly named symbols.
    include_semantic : bool
        Include semantically similar code (EXPENSIVE).
    min_similarity : float
        Minimum similarity threshold.
    timeout : int, optional
        Max seconds to spend (soft limit).
    
    Returns
    -------
    Dict[str, Any]
        Grouped duplicates by type, plus timing metadata.
    """
    p = resolve_selector(project)
    cache = ModidxCacheManager.get(p)
    graph = cache.dependency_graph
    
    if not graph:
        graph = build_graph(str(p))
    
    results = {
        'functional': [],
        'named': [],
        'semantic': []
    }
    
    start = time.monotonic()
    deadline = start + timeout if timeout and timeout > 0 else None
    timed_out = False
    
    if include_functional:
        groups = find_functional_duplicates(graph, min_similarity, deadline=deadline)
        results['functional'] = [g.to_dict() for g in groups]
        timed_out = deadline is not None and time.monotonic() > deadline
    
    if include_named and not timed_out:
        groups = find_named_duplicates(graph, deadline=deadline)
        results['named'] = [g.to_dict() for g in groups]
        timed_out = deadline is not None and time.monotonic() > deadline
    
    if include_semantic and not timed_out:
        groups = find_semantic_duplicates(graph, min_similarity, deadline=deadline)
        results['semantic'] = [g.to_dict() for g in groups]
        timed_out = deadline is not None and time.monotonic() > deadline
    
    total = sum(len(v) for v in results.values())
    elapsed = round(time.monotonic() - start, 2)
    
    return {
        'ok': True,
        'total_groups': total,
        'duplicates': results,
        'timed_out': timed_out,
        'timeout': timeout,
        'elapsed_seconds': elapsed
    }


## MCP Tool Registration

In [ ]:
#| export
def add_dedup_tools(mcp: FastMCP) -> None:
    """Register deduplication tools with MCP server.
    
    Parameters
    ----------
    mcp : FastMCP
        The MCP server instance.
    """
    
    @mcp.tool()
    def find_duplicates(
        project: Optional[str] = None,
        include_functional: bool = True,
        include_named: bool = True,
        similarity_threshold: float = 0.8,
        timeout: Optional[int] = None
    ) -> Dict[str, Any]:
        """Find duplicate code across the project.
        
        Detects:
        - Functionally identical code (same AST)
        - Similarly named symbols (same name, different modules)
        
        NOTE: Semantic analysis requires separate confirmation.
        """
        from nbdev_mcp.utils.config import get_config
        cfg = get_config()
        if timeout is None:
            timeout = cfg.timeout_duplicate_scan
        return find_all_duplicates(
            project,
            include_functional=include_functional,
            include_named=include_named,
            include_semantic=False,  # Requires separate call
            min_similarity=similarity_threshold,
            timeout=timeout
        )
    
    @mcp.tool()
    def find_semantic_duplicates_tool(
        project: Optional[str] = None,
        similarity_threshold: float = 0.85,
        model: str = 'all-MiniLM-L6-v2',
        i_confirm_this_is_expensive: bool = False,
        timeout: Optional[int] = None
    ) -> Dict[str, Any]:
        """Find semantically similar code using embeddings.
        
        EXPENSIVE OPERATION - Uses local embeddings with MPS/CUDA.
        Set i_confirm_this_is_expensive=True to run.
        
        timeout : int, optional
            Max seconds to spend (default from config/environment).
        """
        if not i_confirm_this_is_expensive:
            return {
                'ok': False,
                'error': 'This is an expensive operation. Set i_confirm_this_is_expensive=True to proceed.',
                'device': _get_embedding_device(),
                'model': model
            }
        
        from nbdev_mcp.utils.config import get_config
        cfg = get_config()
        if timeout is None:
            timeout = cfg.timeout_semantic_duplicates

        p = resolve_selector(project)
        cache = ModidxCacheManager.get(p)
        graph = cache.dependency_graph or build_graph(str(p))

        deadline = time.monotonic() + timeout if timeout and timeout > 0 else None
        start = time.monotonic()
        groups = find_semantic_duplicates(graph, similarity_threshold, model, deadline=deadline)
        timed_out = deadline is not None and time.monotonic() > deadline
        elapsed = round(time.monotonic() - start, 2)

        return {
            "ok": True,
            "device": _get_embedding_device(),
            "groups": [g.to_dict() for g in groups],
            "timed_out": timed_out,
            "timeout": timeout,
            "elapsed_seconds": elapsed
        }
    
    @mcp.tool()
    def analyze_duplicate_group(
        project: Optional[str] = None,
        symbols: Optional[List[str]] = None
    ) -> Dict[str, Any]:
        """Deep analysis of a set of potential duplicates.
        
        Provides detailed comparison and merge recommendations.
        """
        if not symbols or len(symbols) < 2:
            return {'ok': False, 'error': 'Need at least 2 symbols to analyze'}
        
        p = resolve_selector(project)
        cache = ModidxCacheManager.get(p)
        graph = cache.dependency_graph or build_graph(str(p))
        
        # Get symbol details
        sym_details = []
        for sym_id in symbols:
            sym = graph.symbols.get(sym_id)
            if sym:
                sym_details.append({
                    'id': sym_id,
                    'name': sym.name,
                    'module': sym.module,
                    'notebook': sym.notebook,
                    'lines': sym.line_count,
                    'hash': sym.source_hash,
                    'imports': len(graph.get_node_edges(sym_id, 'out'))
                })
        
        # Find optimal location
        optimal_loc = find_optimal_location(symbols, graph)
        
        # Gather dependents
        dependents = gather_dependents(symbols, graph)
        
        return {
            'ok': True,
            'symbols': sym_details,
            'optimal_location': optimal_loc,
            'total_dependents': len(dependents),
            'dependents': [d.to_dict() for d in dependents[:20]]  # Limit to 20
        }
    
    @mcp.tool()
    def merge_roadmap(
        project: Optional[str] = None,
        canonical: str = '',
        duplicates: Optional[List[str]] = None,
        target_module: Optional[str] = None
    ) -> Dict[str, Any]:
        """Generate a merge roadmap for duplicate symbols.
        
        Creates step-by-step plan with:
        - Import changes needed
        - Affected code locations
        - Risk assessment
        - Git workflow steps
        
        IMPORTANT: Always wait for human confirmation before executing.
        """
        if not canonical or not duplicates:
            return {'ok': False, 'error': 'Need canonical symbol and duplicates list'}
        
        p = resolve_selector(project)
        cache = ModidxCacheManager.get(p)
        graph = cache.dependency_graph or build_graph(str(p))
        
        # Create a dummy group
        group = DuplicateGroup(
            canonical=canonical,
            duplicates=duplicates,
            similarity_type='manual',
            similarity_score=1.0,
            merge_strategy='keep_canonical'
        )
        
        roadmap = generate_merge_roadmap(group, graph, target_module)
        
        return {
            'ok': True,
            'roadmap': roadmap.to_dict(),
            'warning': 'DO NOT execute merge without human confirmation!'
        }

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()